# Week 1 Lab — First Wikipedia Query and Place-Name Ambiguity

**Course:** Geographic Information Retrieval  
**Week 1 topic:** Course introduction and what makes search geographic?  
**Lab focus:** retrieving a first text document and observing why place names can be ambiguous.

---

## Important note

This version uses the **official MediaWiki API through `requests`** rather than the older `wikipedia` Python package.

Why?  
Some teaching environments may get `JSONDecodeError` when using the old `wikipedia` package, because Wikimedia may return a non-JSON error page if the request is blocked, rate-limited, or missing a descriptive User-Agent.

The code below sets a User-Agent explicitly.

## What you will learn today

By the end of this notebook, you should be able to:

1. run a simple Python notebook from top to bottom;
2. retrieve Wikipedia text through an API request;
3. inspect a page title, URL, and content;
4. switch Wikipedia language editions;
5. observe that place names can be ambiguous;
6. save a small table of retrieved pages for later use.

You are **not** expected to memorize all syntax today.  
The goal is to understand what each step does.

## How to use this notebook

A Jupyter notebook contains two main kinds of cells:

- **Markdown cells**: explanatory text, like this cell.
- **Code cells**: Python code that you can run.

Recommended workflow:

1. Read the markdown cell.
2. Run the code cell below it.
3. Look carefully at the output.
4. Change one small thing and run the cell again.
5. Ask for help if you get an error.

In most notebooks, it is best to run cells **from top to bottom**.

# 1. Import packages

We will use:

- `requests` to call the Wikipedia / MediaWiki API;
- `pandas` to store results in a table;
- `textwrap` only to display long text more neatly.

In [ ]:
# If pandas is missing, you may need:
# !pip install pandas

import requests
import pandas as pd
import textwrap

print("Packages imported successfully.")

# 2. Set a User-Agent

When we make automated requests to Wikipedia, we should identify our script with a descriptive **User-Agent**.

This notebook uses the course contact in the User-Agent so requests are identifiable and not anonymous.

For class use, this is mainly to avoid looking like an anonymous or generic script.

In [ ]:
USER_AGENT = "GIR-Course-Week1/1.0 (University of Vienna course; contact: yingjing.huang@univie.ac.at)"

HEADERS = {
    "User-Agent": USER_AGENT
}

print("User-Agent set to:")
print(USER_AGENT)

# 3. Helper functions

The functions below wrap the MediaWiki API so that the rest of the notebook stays simple.

You do not need to fully understand function definitions today.  
For now, focus on how we use them:

```python
wiki_page("Vienna", language="en")
wiki_search("Cambridge", language="en")
```

In [ ]:
def preview_text(text, n=600, width=90):
    """Print a readable preview of a long text."""
    preview = text[:n]
    print(textwrap.fill(preview, width=width))
    if len(text) > n:
        print("\n[...]")


def wiki_api_get(language, params):
    """Send a GET request to the MediaWiki API and return JSON data."""
    url = f"https://{language}.wikipedia.org/w/api.php"

    response = requests.get(
        url,
        params=params,
        headers=HEADERS,
        timeout=20
    )

    if response.status_code != 200:
        print("HTTP status code:", response.status_code)
        print("First part of response:")
        print(response.text[:500])

    response.raise_for_status()
    return response.json()


def wiki_page(title, language="en"):
    """Retrieve a Wikipedia page title, URL, and plain-text extract."""
    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts|info",
        "explaintext": 1,
        "inprop": "url",
        "redirects": 1,
        "titles": title
    }

    data = wiki_api_get(language, params)
    pages = data["query"]["pages"]
    page = next(iter(pages.values()))

    if "missing" in page:
        raise ValueError(f"No Wikipedia page found for: {title}")

    return {
        "query": title,
        "language": language,
        "title": page.get("title", ""),
        "url": page.get("fullurl", ""),
        "content": page.get("extract", "")
    }


def wiki_search(query, language="en", limit=10):
    """Search Wikipedia and return a list of candidate page titles."""
    params = {
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": query,
        "srlimit": limit
    }

    data = wiki_api_get(language, params)
    return [item["title"] for item in data["query"]["search"]]

# 4. First query: retrieve a Wikipedia page

We start with a very simple example:

```python
wiki_page("Vienna", language="en")
```

Here:

- `"Vienna"` is the **input place name**;
- `"en"` means English Wikipedia;
- the result contains a title, URL, and text content.

In GIR terms, we are doing the first tiny step:

```text
place name → Wikipedia page → text document
```

In [ ]:
topic = "Vienna"

page = wiki_page(topic, language="en")

print("Topic:", topic)
print("Returned page title:", page["title"])
print("URL:", page["url"])
print("Number of characters:", len(page["content"]))
print("\nPreview:\n")
preview_text(page["content"], n=700)

## Think about the output

After running the cell above, answer these questions for yourself:

1. Is the returned result structured like a table, or is it mostly free text?
2. Which words in the text are geographic references?
3. Does the text only mention Vienna, or also other places?
4. Why might this kind of text be useful for a GIR system?

# 5. Inspect title, URL, and content

In a simple retrieval system, each document often needs:

| Field | Why it matters |
|---|---|
| title | helps identify the document |
| URL | gives provenance / source |
| content | provides the text to be searched and analyzed |

Later in the course, we will use text content for geoparsing, geocoding, indexing, and ranking.

In [ ]:
print("Title:")
print(page["title"])

print("\nURL:")
print(page["url"])

print("\nFirst 1000 characters of content:")
preview_text(page["content"], n=1000)

# 6. Store one retrieved page in a table

Search systems usually do not work with one document only.  
They work with a **collection** of documents.

We will start a very small document table.

In [ ]:
documents = []

documents.append({
    "place_query": topic,
    "language": "en",
    "title": page["title"],
    "url": page["url"],
    "preview": page["content"][:500],
    "note": "English Wikipedia page for Vienna."
})

df_documents = pd.DataFrame(documents)
df_documents

# 7. Try another place

Now change the place name and run a similar query.

Examples you can try:

- `"Salzburg"`
- `"Innsbruck"`
- `"Graz"`
- `"Danube"`
- `"Schönbrunn Palace"`
- `"University of Vienna"`

Choose one place that is interesting to you.

In [ ]:
# TODO: change this place name.
my_place = "Salzburg"

my_page = wiki_page(my_place, language="en")

print("Title:", my_page["title"])
print("URL:", my_page["url"])
print("\nPreview:\n")
preview_text(my_page["content"], n=700)

In [ ]:
documents.append({
    "place_query": my_place,
    "language": "en",
    "title": my_page["title"],
    "url": my_page["url"],
    "preview": my_page["content"][:500],
    "note": "My own selected English Wikipedia page."
})

df_documents = pd.DataFrame(documents)
df_documents

# 8. Same place, different language

One important issue in GIR is that the same place can appear under different names.

For example:

| English | German | French | Chinese |
|---|---|---|---|
| Vienna | Wien | Vienne | 维也纳 |

Wikipedia has different language editions.  
We can use a different language code:

```python
wiki_page("Wien", language="de")
```

In [ ]:
german_query = "Wien"

german_page = wiki_page(german_query, language="de")

print("German page title:", german_page["title"])
print("German page URL:", german_page["url"])
print("\nPreview:\n")
preview_text(german_page["content"], n=700)

In [ ]:
documents.append({
    "place_query": german_query,
    "language": "de",
    "title": german_page["title"],
    "url": german_page["url"],
    "preview": german_page["content"][:500],
    "note": "German Wikipedia page for Vienna/Wien."
})

df_documents = pd.DataFrame(documents)
df_documents

# 9. Place-name ambiguity

A major problem in GIR is that one name can refer to many possible places.

Examples:

- `Cambridge` can refer to Cambridge, UK or Cambridge, Massachusetts.
- `Georgia` can refer to a country or a U.S. state.
- `Java` can refer to an island, a programming language, or coffee.
- `Jordan` can refer to a country, a river, or a person name.

Before geocoding a place name, we often need to know **which place is intended**.

## Search candidates

The function `wiki_search(...)` returns a list of possible page titles.

This is useful for seeing ambiguity.

In [ ]:
ambiguous_query = "Cambridge"

search_results = wiki_search(ambiguous_query, language="en", limit=10)

print("Search results for:", ambiguous_query)
for i, result in enumerate(search_results, start=1):
    print(f"{i}. {result}")

## Try another ambiguous name

Change the query below to one of:

- `"Georgia"`
- `"Java"`
- `"Jordan"`
- `"Washington"`
- `"Paris"`
- `"Baden"`
- or another ambiguous place name you know.

In [ ]:
# TODO: change this ambiguous query.
my_ambiguous_query = "Georgia"

my_search_results = wiki_search(my_ambiguous_query, language="en", limit=10)

print("Search results for:", my_ambiguous_query)
for i, result in enumerate(my_search_results, start=1):
    print(f"{i}. {result}")

## Discussion

Look at the search results above.

Answer briefly:

1. Are all returned pages places?
2. If there are multiple places, which one would be the default?
3. What context would help choose the intended place?
4. Why might this be hard for a computer?

This issue will return in Week 4 when we study **geocoding**.

# 10. Your own mini task

Now you will retrieve one page of your choice.

Choose a place that is interesting to you.  
It can be a city, region, river, landmark, university, or neighborhood.

Fill in the variables below.

In [ ]:
# TODO: edit these values.
student_place_query = "Innsbruck"
student_language = "en"
student_note = "A city in Austria; may be useful for a small geographic text corpus."

student_page = wiki_page(student_place_query, language=student_language)

print("Title:", student_page["title"])
print("URL:", student_page["url"])
print("\nPreview:\n")
preview_text(student_page["content"], n=700)

In [ ]:
documents.append({
    "place_query": student_place_query,
    "language": student_language,
    "title": student_page["title"],
    "url": student_page["url"],
    "preview": student_page["content"][:500],
    "note": student_note
})

df_documents = pd.DataFrame(documents)
df_documents

# 11. Save the table

The table can be saved as a CSV file.

CSV means **comma-separated values**.  
It is a simple file format that can be opened by Excel, Python, or text editors.

We will save it into the current notebook folder.

In [ ]:
output_file = "week1_wikipedia_documents.csv"

df_documents.to_csv(output_file, index=False, encoding="utf-8")

print("Saved:", output_file)
print("Number of documents:", len(df_documents))

# 12. Reflection

Write short answers below.  
You can answer in English or German. Chinese is also fine if needed, but English is preferred for course communication.

## Questions

1. Which place did you choose?
2. Did the page contain other geographic references?
3. Did you observe any ambiguity or language issue?
4. Why might this text be useful for a GIR system?

Double click this markdown cell and type your answers below.

---

**Your answers:**

1. 
2. 
3. 
4.

# 13. Optional extension

This part is optional. Do it only if you finish early.

## Option A: compare two languages

Retrieve the same place in two different languages and compare:

- title
- URL
- first paragraph
- place names mentioned

## Option B: build a tiny corpus

Retrieve 5 pages related to one topic, for example:

- Austrian cities
- rivers in Europe
- landmarks in Vienna
- universities in Europe

Then save all pages in one CSV file.

In [ ]:
# OPTIONAL: build a tiny corpus of several places.

places = ["Vienna", "Salzburg", "Innsbruck", "Graz", "Linz"]

tiny_corpus = []

for place_name in places:
    try:
        p = wiki_page(place_name, language="en")
        tiny_corpus.append({
            "place_query": place_name,
            "language": "en",
            "title": p["title"],
            "url": p["url"],
            "preview": p["content"][:500],
            "note": "Optional tiny corpus example."
        })
        print("Retrieved:", p["title"])

    except Exception as error:
        print("Could not retrieve:", place_name)
        print("Error:", error)

df_tiny_corpus = pd.DataFrame(tiny_corpus)
df_tiny_corpus

# 14. Summary

Today you completed the first technical step in the course:

```text
place name → Wikipedia page → text document
```

You also saw several ideas that will return later:

| Idea | Later course topic |
|---|---|
| text document | information retrieval |
| place name | geoparsing |
| ambiguous place name | geocoding / disambiguation |
| same place in different languages | gazetteers and place-name variants |
| document table | indexing and retrieval |

Next week, we will ask:

> How can a system automatically find place names in text?

That topic is called **geoparsing**.